# Diagrama de Sankey: Fluxo Metodológico da Literatura Científica
## Rastreamento de Camadas de Entrada $\longrightarrow$ Modelos de IA/DL $\longrightarrow$ Saídas e Alvos Minerais

Este notebook realiza a extração, balanceamento e visualização dos fluxos metodológicos dos **104 artigos científicos** analisados no projeto.

**Novidades desta versão:**
- **Renderizador resiliente**: Elimina erros de renderização (`fig.show()`) no VS Code e Jupyter com fallback automático para HTML embutido.
- **Visualização Com Rótulos**: Exibição clássica com os nomes dos nós fixados nas barras.
- **Visualização Sem Rótulos**: Estilo minimalista com foco nas linhas de fluxo (ideal para apresentações limpas, com informações completas no *hover* ao passar o mouse).

---

### Célula 1: Configuração do Ambiente e Importação de Bibliotecas

In [1]:
# [Célula 1] Importação das bibliotecas essenciais
import os
import sys
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML

# Configuração de encoding para ambientes Windows
if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

# Configuração do renderer do Plotly para compatibilidade total no VS Code e Jupyter
for r in ['notebook_connected', 'vscode', 'iframe', 'html']:
    if r in pio.renderers:
        pio.renderers.default = r
        break

print('Bibliotecas carregadas com sucesso!')
print(f'Pandas versão: {pd.__version__}')
print(f'Renderer Plotly ativo: {pio.renderers.default}')

Bibliotecas carregadas com sucesso!
Pandas versão: 2.2.3
Renderer Plotly ativo: notebook_connected


### Célula 2: Carregamento e Inspeção dos Dados Consolidados
Carrega o arquivo `sintese_entrada_modelo_saida.csv` (ou a base consolidada de classificação e SOTA).

In [2]:
# [Célula 2] Carregamento da base de dados
caminho_csv = Path('sintese_entrada_modelo_saida.csv')
if not caminho_csv.exists():
    caminho_csv = Path('base_consolidada_classificacao_e_sota.csv')
    
if not caminho_csv.exists():
    raise FileNotFoundError('Nenhum arquivo consolidado encontrado no diretório!')

df = pd.read_csv(caminho_csv, sep=';', encoding='utf-8-sig')
print(f'-> Base carregada com sucesso: {len(df)} artigos.')
print(f'-> Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas.')

# Prévia das colunas essenciais
cols_previa = [c for c in df.columns if any(k in c.lower() for k in ['rank', 'classific', 'síntese', 'sintese'])]
df[cols_previa].head(3)

-> Base carregada com sucesso: 104 artigos.
-> Dimensões: 104 linhas x 34 colunas.


,ID: Rank,ID: Classificação,SÍNTESE: Entrada,SÍNTESE: Modelo,SÍNTESE: Saída
0,1,★★★ Excelente,"satélite; aérea; sintética (ASTER, Hiperespect...",VAE-BIRCH; deep embedded clustering; convoluti...,Alvo: minério/mineral
1,2,★★★ Excelente,"satélite (ASTER, Hiperespectral)",EndNet | Base: outro | Loss: Spectral Angle Di...,Alvo: minério/mineral | Métricas: Receiver ope...
2,3,★★☆ Alta,"[Termos RS]: aster, remote sensing, multispect...","[Termos DL]: vit, artificial intelligence","[Termos Prospecção]: mineral prospect, mineral..."


### Célula 3: Função de Extração das Dimensões Metodológicas
Classifica cada artigo nas dimensões padronizadas de **Entrada**, **Modelo** e **Saída**.

In [3]:
# [Célula 3] Classificação de dimensões por artigo
def extrair_dimensoes_artigo(r: pd.Series):
    """
    Classifica o artigo em: (camadas, modelos, saidas).
    """
    # 1. ENTRADA (Camadas de dados e sensores)
    texto_in = " ".join([
        str(r.get("ENTRADA: Fontes e Sensores", "")),
        str(r.get("SÍNTESE: Entrada", "")),
        str(r.get("ID: Título do Artigo", "")),
        str(r.get("Termos Sensoriamento Remoto", "")),
    ]).lower()
    
    camadas = []
    if "aster" in texto_in or str(r.get("ENTRADA: Usa ASTER", "")).lower() in ["sim", "true", "yes"]:
        camadas.append("ASTER (SWIR/TIR)")
    if any(k in texto_in for k in ["hiperespectral", "hyperspectral", "prisma", "aviris", "hyperion"]) or \
       str(r.get("ENTRADA: Hiperespectral", "")).lower() in ["sim", "true", "yes"]:
        camadas.append("Hiperespectral (PRISMA/AVIRIS)")
    if any(k in texto_in for k in ["landsat", "sentinel-2", "sentinel 2", "worldview", "spot", "multiespectral", "multispectral"]):
        camadas.append("Óptico (Landsat/Sentinel)")
    if any(k in texto_in for k in ["sar", "radar", "sentinel-1", "palsar"]):
        camadas.append("Radar SAR")
    if any(k in texto_in for k in ["geoquím", "geochemi", "sedimento", "stream sediment", "soil"]):
        camadas.append("Geoquímica")
    if any(k in texto_in for k in ["geofís", "geophysic", "aeromagnet", "magnetometria", "gamma", "gama", "gravimetr"]):
        camadas.append("Aerogeofísica")
    if any(k in texto_in for k in ["drone", "uav", "vant"]):
        camadas.append("Drone / VANT")
        
    if not camadas:
        camadas.append("Satélite Convencional")

    # 2. MODELO (Arquiteturas computacionais)
    texto_mod = " ".join([
        str(r.get("MODELO: Modelos DL", "")),
        str(r.get("MODELO: Arquitetura Base", "")),
        str(r.get("SÍNTESE: Modelo", "")),
        str(r.get("ID: Título do Artigo", "")),
        str(r.get("Termos Deep Learning", "")),
    ]).lower()
    
    modelos = []
    if any(k in texto_mod for k in ["cnn", "convolutional", "lenet", "resnet", "densenet", "vgg", "1d-cnn", "2d-cnn", "3d-cnn", "fcnn"]):
        modelos.append("Redes Convolucionais (CNNs)")
    if any(k in texto_mod for k in ["u-net", "unet", "fcn", "segmentation", "segmentação"]):
        modelos.append("U-Net / Segmentação")
    if any(k in texto_mod for k in ["transformer", "vit", "vision transformer", "attention", "atenção", "mamba"]):
        modelos.append("Transformers / ViT")
    if any(k in texto_mod for k in ["gcn", "gnn", "graph convolutional", "redes em grafo"]):
        modelos.append("Graph Networks (GCN)")
    if any(k in texto_mod for k in ["autoencoder", "dae", "vae", "auto-encoder", "deep autoencoder"]):
        modelos.append("Autoencoders (DAE/VAE)")
    if any(k in texto_mod for k in ["random forest", "svm", "support vector", "xgboost", "catboost", "ensemble"]):
        modelos.append("Machine Learning Clássico")
    if any(k in texto_mod for k in ["mlp", "multilayer perceptron", "ann", "artificial neural network"]):
        modelos.append("Redes Neurais Densas (MLP)")
    if any(k in texto_mod for k in ["gan", "generative adversarial", "wgan"]):
        modelos.append("Redes Generativas (GAN)")
        
    if not modelos:
        modelos.append("Outros Modelos IA")

    # 3. SAÍDA (Alvos minerais e produtos)
    texto_out = " ".join([
        str(r.get("SAÍDA: Objeto Prospecção", "")),
        str(r.get("SÍNTESE: Saída", "")),
        str(r.get("ID: Título do Artigo", "")),
        str(r.get("Termos Prospecção Mineral", "")),
    ]).lower()
    
    saidas = []
    if "fosfat" in texto_out or "phosphate" in texto_out or str(r.get("SAÍDA: Prospecta Fosfato", "")).lower() in ["sim", "true", "yes"]:
        saidas.append("Fosfato")
    if any(k in texto_out for k in ["cobre", "copper", "porphyry cu", "skarn"]):
        saidas.append("Cobre (Pórfiro/Skarn)")
    if any(k in texto_out for k in ["ouro", "gold", "orogenic", "epithermal"]):
        saidas.append("Ouro (Orogênico/Epitermal)")
    if any(k in texto_out for k in ["alteração hidrotermal", "hydrothermal alteration", "alteration"]):
        saidas.append("Alteração Hidrotermal")
    if any(k in texto_out for k in ["litológ", "litholog", "geological mapping", "mapeamento geológico", "rochas"]):
        saidas.append("Mapeamento Litológico")
    if any(k in texto_out for k in ["lítio", "lithium", "pegmatit"]):
        saidas.append("Lítio (Pegmatitos)")
    if any(k in texto_out for k in ["ferro", "iron ore", "bif"]):
        saidas.append("Minério de Ferro")
    if any(k in texto_out for k in ["zinco", "chumbo", "pb-zn", "nickel", "níquel", "polymetallic", "polimetál"]):
        saidas.append("Polimetálicos (Pb-Zn/Ni)")
        
    if not saidas:
        saidas.append("Prospecção Mineral Geral")
        
    return camadas, modelos, saidas

print('Função extrair_dimensoes_artigo definida com sucesso!')

Função extrair_dimensoes_artigo definida com sucesso!


### Célula 4: Cálculo e Balanceamento dos Fluxos do Sankey
Mapeia as co-ocorrências ponderadas garantindo **conservação balanceada** em todos os nós.

In [4]:
# [Célula 4] Agregação de fluxos ponderados
fluxos_entrada_modelo = defaultdict(float)
fluxos_modelo_saida = defaultdict(float)
rotas_completas = defaultdict(float)

for _, r in df.iterrows():
    camadas, modelos, saidas = extrair_dimensoes_artigo(r)
    n_c, n_m, n_s = len(camadas), len(modelos), len(saidas)
    
    peso_in_mod = 1.0 / (n_c * n_m)
    peso_mod_out = 1.0 / (n_m * n_s)
    
    for c in camadas:
        for m in modelos:
            fluxos_entrada_modelo[(c, m)] += peso_in_mod
            
    for m in modelos:
        for s in saidas:
            fluxos_modelo_saida[(m, s)] += peso_mod_out

    for c in camadas:
        for m in modelos:
            for s in saidas:
                rotas_completas[(c, m, s)] += 1.0 / (n_c * n_m * n_s)

print(f'-> Total de conexões Entrada -> Modelo: {len(fluxos_entrada_modelo)}')
print(f'-> Total de conexões Modelo -> Saída  : {len(fluxos_modelo_saida)}')
print(f'-> Volume conservado: {sum(fluxos_entrada_modelo.values()):.1f} artigos equivalentes.')

-> Total de conexões Entrada -> Modelo: 50
-> Total de conexões Modelo -> Saída  : 43
-> Volume conservado: 104.0 artigos equivalentes.


### Célula 5: Mapeamento de Nós, Links e Paleta de Cores
Cria as listas de nós e links com cores coordenadas para cada estágio.

In [5]:
# [Célula 5] Indexação e definição de cores
nos_entrada = sorted(list({c for c, m in fluxos_entrada_modelo.keys()}))
nos_modelo = sorted(list({m for c, m in fluxos_entrada_modelo.keys()} | {m for m, s in fluxos_modelo_saida.keys()}))
nos_saida = sorted(list({s for m, s in fluxos_modelo_saida.keys()}))

todos_nos_rotulos = []
mapa_indices = {}

for n in nos_entrada:
    mapa_indices[f'IN_{n}'] = len(todos_nos_rotulos)
    todos_nos_rotulos.append(f'[Entrada] {n}')
    
for n in nos_modelo:
    mapa_indices[f'MOD_{n}'] = len(todos_nos_rotulos)
    todos_nos_rotulos.append(f'[Modelo] {n}')
    
for n in nos_saida:
    mapa_indices[f'OUT_{n}'] = len(todos_nos_rotulos)
    todos_nos_rotulos.append(f'[Saída] {n}')

# Cores dos nós
cores_entrada = ['#16A085', '#1ABC9C', '#2ECC71', '#27AE60', '#48C9B0', '#76D7C4', '#117864', '#0E6251']
cores_modelo = ['#2980B9', '#3498DB', '#5DADE2', '#1F618D', '#2874A6', '#154360', '#2471A3', '#5499C7']
cores_saida = ['#D35400', '#E67E22', '#F39C12', '#CA6F1E', '#BA4A00', '#873600', '#E59866', '#DC7633', '#B9770E']

cores_nos = []
for i, n in enumerate(nos_entrada):
    cores_nos.append(cores_entrada[i % len(cores_entrada)])
for i, n in enumerate(nos_modelo):
    cores_nos.append(cores_modelo[i % len(cores_modelo)])
for i, n in enumerate(nos_saida):
    cores_nos.append(cores_saida[i % len(cores_saida)])

# Links translúcidos
sources, targets, values, cores_links = [], [], [], []

for (c, m), peso in fluxos_entrada_modelo.items():
    if peso > 0.05:
        src_idx = mapa_indices[f'IN_{c}']
        tgt_idx = mapa_indices[f'MOD_{m}']
        sources.append(src_idx)
        targets.append(tgt_idx)
        values.append(round(peso, 2))
        hex_cor = cores_nos[src_idx].lstrip('#')
        r_c, g_c, b_c = tuple(int(hex_cor[j:j+2], 16) for j in (0, 2, 4))
        cores_links.append(f'rgba({r_c}, {g_c}, {b_c}, 0.40)')

for (m, s), peso in fluxos_modelo_saida.items():
    if peso > 0.05:
        src_idx = mapa_indices[f'MOD_{m}']
        tgt_idx = mapa_indices[f'OUT_{s}']
        sources.append(src_idx)
        targets.append(tgt_idx)
        values.append(round(peso, 2))
        hex_cor = cores_nos[src_idx].lstrip('#')
        r_c, g_c, b_c = tuple(int(hex_cor[j:j+2], 16) for j in (0, 2, 4))
        cores_links.append(f'rgba({r_c}, {g_c}, {b_c}, 0.40)')

print(f'Nós preparados: {len(todos_nos_rotulos)} | Conexões: {len(sources)}')

Nós preparados: 25 | Conexões: 93


### Célula 6: Função Construtora e Exibidora do Sankey (Com e Sem Rótulos)
Esta função aceita o parâmetro `com_rotulos=True` ou `com_rotulos=False`:
- **`com_rotulos=True`**: Exibe o texto de cada nó nas barras laterais.
- **`com_rotulos=False`**: Oculta o texto estático dos nós, gerando uma visualização limpa e despoluída dos fluxos (os detalhes aparecem interativamente ao passar o mouse).

In [6]:
def criar_figura_sankey(
    modo: str = 'com_rotulos',
    proximidade_colunas: float = 0.70,
    largura: int = 1080,
    altura: int = 760
) -> go.Figure:
    """
    Cria o Diagrama de Sankey com posicionamento horizontal personalizado.
    proximidade_colunas: valor entre 0.40 (muito próximas) e 1.0 (extremidades da tela).
    """
    # Calcula posições horizontais x para aproximar as colunas
    prox = max(0.40, min(1.0, proximidade_colunas))
    x_in = round(0.5 - prox / 2.0, 3)
    x_mod = 0.50
    x_out = round(0.5 + prox / 2.0, 3)

    node_x = []
    for n in todos_nos_rotulos:
        if n.startswith('[Entrada]'):
            node_x.append(x_in)
        elif n.startswith('[Modelo]'):
            node_x.append(x_mod)
        else:
            node_x.append(x_out)

    # Configura rótulos e espaçamento de acordo com o modo
    if modo == 'rotulos_fora':
        rotulos_exibicao = todos_nos_curtos
        subtitulo = f'Rótulos Externos Despoluídos | Proximidade: {int(prox*100)}%'
        espessura_no = 18
        espaco_pad = 22
        margens = dict(l=80, r=80, t=110, b=40)
    elif modo == 'com_rotulos':
        rotulos_exibicao = todos_nos_rotulos
        subtitulo = f'Colunas Aproximadas com Rótulos nos Nós | Proximidade: {int(prox*100)}%'
        espessura_no = 22
        espaco_pad = 18
        margens = dict(l=40, r=40, t=110, b=35)
    else:  # sem_rotulos
        rotulos_exibicao = ["" for _ in todos_nos_rotulos]
        subtitulo = f'Visão Minimalista (Hover Interativo) | Proximidade: {int(prox*100)}%'
        espessura_no = 15
        espaco_pad = 20
        margens = dict(l=40, r=40, t=110, b=35)

    fig = go.Figure(data=[go.Sankey(
        arrangement='snap',
        node=dict(
            pad=espaco_pad,
            thickness=espessura_no,
            line=dict(color='#2C3E50', width=0.8),
            label=rotulos_exibicao,
            customdata=todos_nos_rotulos,
            color=cores_nos,
            x=node_x,  # Posições horizontais calculadas
            hovertemplate='<b>%{customdata}</b><br>Fluxo Ponderado: %{value:.1f} artigos<extra></extra>'
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=cores_links,
            hovertemplate='<b>%{source.customdata}</b>  →  <b>%{target.customdata}</b><br>Volume: %{value:.2f} artigos<extra></extra>'
        )
    )])

    # Cabeçalhos no topo centralizados dinamicamente sobre cada coluna aproximada
    anotacoes = [
        dict(x=x_in, y=1.07, xref='paper', yref='paper', text='<b>🟩 1. ENTRADA</b>', showarrow=False, font=dict(size=12, color='#16A085'), xanchor='center'),
        dict(x=x_mod, y=1.07, xref='paper', yref='paper', text='<b>🟦 2. MODELOS</b>', showarrow=False, font=dict(size=12, color='#2980B9'), xanchor='center'),
        dict(x=x_out, y=1.07, xref='paper', yref='paper', text='<b>🟧 3. SAÍDAS</b>', showarrow=False, font=dict(size=12, color='#D35400'), xanchor='center'),
    ]

    fig.update_layout(
        title=dict(
            text=f'<b>DIAGRAMA DE SANKEY: FLUXO METODOLÓGICO</b><br>'
                 f'<span style="font-size: 13px; color: #566573;">Camadas de Entrada → Modelos de IA → Saídas | {subtitulo} (N = 104 artigos)</span>',
            font=dict(size=18, family='Segoe UI, Arial, sans-serif', color='#1A252F'),
            x=0.03,
            y=0.98
        ),
        annotations=anotacoes,
        font=dict(size=10.5, family='Segoe UI, Arial, sans-serif', color='#2C3E50'),
        height=altura,
        width=largura,
        margin=margens,
        paper_bgcolor='#FFFFFF',
        plot_bgcolor='#FFFFFF'
    )
    return fig


def renderizar_seguro(fig: go.Figure):
    """Exibe o gráfico com fallback transparente para HTML embutido, evitando erros de nbformat."""
    try:
        fig.show()
    except Exception as erro:
        html_code = fig.to_html(include_plotlyjs='cdn', full_html=False)
        display(HTML(html_code))


### Célula 7: Renderização da Versão COM RÓTULOS (Visão Completa)
Exibe o diagrama tradicional com os nomes de todos os nós visíveis nas barras.

In [7]:
# [Célula 7] Versão COM RÓTULOS e COLUNAS APROXIMADAS
fig_com_rotulos = criar_figura_sankey(
    modo='com_rotulos',
    proximidade_colunas=0.20,  # Experimente 0.65 ou 0.60 para aproximar ainda mais
    largura=1080
)
renderizar_seguro(fig_com_rotulos)


### Célula 8: Renderização da Versão SEM RÓTULOS (Visão Minimalista)
Exibe o diagrama despoluído focado na fluidez visual dos caminhos. **Passe o mouse sobre qualquer nó ou linha** para inspecionar os detalhes.

In [8]:
# [Célula 8] Versão SEM RÓTULOS (Minimalista)
fig_sem_rotulos = criar_figura_sankey(com_rotulos=False)
renderizar_seguro(fig_sem_rotulos)

TypeError: criar_figura_sankey() got an unexpected keyword argument 'com_rotulos'

### Célula 9: Exportação dos Gráficos em Disco (HTML e PNG)
Salva tanto a versão **com rótulos** quanto a versão **sem rótulos** nos formatos `.html` e `.png`.

In [ ]:
# [Célula 9] Exportação em disco
pasta_saida = Path('graficos')
pasta_saida.mkdir(parents=True, exist_ok=True)

# 1. Salva versões COM rótulos
html_com = pasta_saida / 'sankey_com_rotulos.html'
png_com = pasta_saida / 'sankey_com_rotulos.png'
fig_com_rotulos.write_html(str(html_com), include_plotlyjs='cdn')
try:
    fig_com_rotulos.write_image(str(png_com), scale=2.5)
    print(f'✓ Imagem PNG (com rótulos) salva: {png_com.resolve()}')
except Exception as e:
    print(f'Aviso PNG (com rótulos): {e}')
print(f'✓ HTML interativo (com rótulos) salvo: {html_com.resolve()}')

# 2. Salva versões SEM rótulos
html_sem = pasta_saida / 'sankey_sem_rotulos.html'
png_sem = pasta_saida / 'sankey_sem_rotulos.png'
fig_sem_rotulos.write_html(str(html_sem), include_plotlyjs='cdn')
try:
    fig_sem_rotulos.write_image(str(png_sem), scale=2.5)
    print(f'✓ Imagem PNG (sem rótulos) salva: {png_sem.resolve()}')
except Exception as e:
    print(f'Aviso PNG (sem rótulos): {e}')
print(f'✓ HTML interativo (sem rótulos) salvo: {html_sem.resolve()}')

✓ Imagem PNG (com rótulos) salva: C:\Users\Luisl\OneDrive\Documentos\GitHub\AnaliseArtigos\graficos\sankey_com_rotulos.png
✓ HTML interativo (com rótulos) salvo: C:\Users\Luisl\OneDrive\Documentos\GitHub\AnaliseArtigos\graficos\sankey_com_rotulos.html
✓ Imagem PNG (sem rótulos) salva: C:\Users\Luisl\OneDrive\Documentos\GitHub\AnaliseArtigos\graficos\sankey_sem_rotulos.png
✓ HTML interativo (sem rótulos) salvo: C:\Users\Luisl\OneDrive\Documentos\GitHub\AnaliseArtigos\graficos\sankey_sem_rotulos.html


### Célula 10: Análise Estatística das Top Rotas Metodológicas
Exibe a tabela com os 10 fluxos triplos mais frequentes da literatura.

In [ ]:
# [Célula 10] Tabela de principais rotas
top_rotas = sorted(rotas_completas.items(), key=lambda x: x[1], reverse=True)[:10]

linhas = []
for idx_r, ((c, m, s), v) in enumerate(top_rotas, 1):
    linhas.append({
        'Posição': idx_r,
        'Camada de Entrada': c,
        'Modelo Computacional': m,
        'Saída / Alvo': s,
        'Artigos Eq.': round(v, 2),
        '% do Total': f'{v/len(df)*100:.1f}%'
    })

df_top_rotas = pd.DataFrame(linhas)
print('=== TOP 10 ROTAS METODOLÓGICAS MAIS FREQUENTES ===')
df_top_rotas

=== TOP 10 ROTAS METODOLÓGICAS MAIS FREQUENTES ===


,Posição,Camada de Entrada,Modelo Computacional,Saída / Alvo,Artigos Eq.,% do Total
0,1,Hiperespectral (PRISMA/AVIRIS),Redes Convolucionais (CNNs),Prospecção Mineral Geral,10.50,10.1%
1,2,Satélite Convencional,Transformers / ViT,Prospecção Mineral Geral,7.33,7.1%
2,3,Hiperespectral (PRISMA/AVIRIS),Transformers / ViT,Prospecção Mineral Geral,6.25,6.0%
3,4,Satélite Convencional,Redes Convolucionais (CNNs),Prospecção Mineral Geral,3.83,3.7%
4,5,Hiperespectral (PRISMA/AVIRIS),Redes Convolucionais (CNNs),Mapeamento Litológico,3.50,3.4%
5,6,Hiperespectral (PRISMA/AVIRIS),Machine Learning Clássico,Mapeamento Litológico,2.92,2.8%
6,7,ASTER (SWIR/TIR),Redes Convolucionais (CNNs),Prospecção Mineral Geral,2.83,2.7%
7,8,Radar SAR,Transformers / ViT,Prospecção Mineral Geral,2.58,2.5%
8,9,Radar SAR,Redes Convolucionais (CNNs),Prospecção Mineral Geral,2.50,2.4%
9,10,ASTER (SWIR/TIR),Redes Convolucionais (CNNs),Ouro (Orogênico/Epitermal),2.50,2.4%
